# Session 8: Self-Reflection & Critique
## Building AI Systems That Can Evaluate and Improve Themselves

**BIA School of Technology & AI**  
Generative AI & Agentic AI Development  

---

In this session, we explore **self-reflection** and **critique mechanisms** that enable AI systems to:
- Generate content and then evaluate their own work
- Iteratively refine outputs based on feedback
- Use structured reasoning to identify and fix issues
- Judge and compare multiple approaches

These patterns are fundamental to building agentic systems that improve over time without human intervention.

## Setup

Install and import required libraries. Make sure you have a Google API key set as `GOOGLE_API_KEY` environment variable.

In [ ]:
# !pip install langchain-google-genai langchain-core langgraph pydantic

import os
import json
from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# Initialize the LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.0,
    max_tokens=1500
)

print("✓ Setup complete. Ready to explore self-reflection and critique!")

---
# Section 1: Basic Self-Critique

The simplest pattern: **Generate → Critique → Done**

We'll generate a response, then have another chain analyze it for issues.

In [ ]:
# Define Pydantic schema for critique results
class CritiqueResult(BaseModel):
    """Structured output for a critique of generated content."""
    issues: List[str] = Field(
        ..., 
        description="List of specific issues found (bugs, style, clarity, etc.)"
    )
    overall_score: int = Field(
        ..., 
        description="Overall quality score from 1-10"
    )
    suggestions: List[str] = Field(
        ..., 
        description="Actionable suggestions for improvement"
    )

print("✓ CritiqueResult schema defined")

In [ ]:
# Chain 1: Generate a response
generation_prompt = ChatPromptTemplate.from_template(
    "Generate a {language} implementation of {task}. Keep it brief but functional."
)

generate_chain = generation_prompt | llm | StrOutputParser()

# Example: generate a Python function
generated_code = generate_chain.invoke({
    "language": "Python",
    "task": "a function that checks if a number is prime"
})

print("Generated Code:")
print("=" * 60)
print(generated_code)
print("=" * 60)

In [ ]:
# Chain 2: Critique the generated code
critique_prompt = ChatPromptTemplate.from_template(
    """Review the following code for bugs, style issues, and clarity problems.
    
Code to review:
{code}

Provide your critique as JSON with the following structure:
{{
  "issues": [list of specific problems found],
  "overall_score": [1-10 score],
  "suggestions": [list of improvements]
}}

Return only valid JSON, no other text.
"""
)

critique_chain = critique_prompt | llm | StrOutputParser()

# Get critique
critique_response = critique_chain.invoke({"code": generated_code})

# Parse JSON
critique_json = json.loads(critique_response)
critique_result = CritiqueResult(**critique_json)

print("\nCritique Results:")
print("=" * 60)
print(f"Overall Score: {critique_result.overall_score}/10")
print(f"\nIssues Found:")
for i, issue in enumerate(critique_result.issues, 1):
    print(f"  {i}. {issue}")
print(f"\nSuggestions:")
for i, sugg in enumerate(critique_result.suggestions, 1):
    print(f"  {i}. {sugg}")
print("=" * 60)

---
# Section 2: Iterative Self-Refine

Move beyond single critique: **Generate → Feedback → Refine → Repeat**

Key stopping criteria:
- Score >= 8 (good enough)
- Max 3 iterations (time/cost)
- No new issues (diminishing returns)

In [ ]:
# Enhanced Pydantic schemas for refinement loop
class Feedback(BaseModel):
    """Feedback with actionable steps."""
    score: int = Field(..., description="Quality score 1-10")
    issues: List[str] = Field(..., description="Specific problems")
    actionable_suggestions: List[str] = Field(
        ..., 
        description="Concrete next steps to improve"
    )

class RefinedOutput(BaseModel):
    """The refined/improved output."""
    content: str = Field(..., description="The improved content")
    change_summary: str = Field(
        ..., 
        description="Summary of what changed"
    )

print("✓ Feedback and RefinedOutput schemas defined")

In [ ]:
# Setup: Feedback chain
feedback_prompt = ChatPromptTemplate.from_template(
    """Analyze this text and provide constructive feedback:
    
Text:
{text}

Return JSON with this structure:
{{
  "score": [1-10],
  "issues": [list of problems],
  "actionable_suggestions": [concrete improvements]
}}

Return only valid JSON.
"""
)

feedback_chain = feedback_prompt | llm | StrOutputParser()

# Setup: Refinement chain
refine_prompt = ChatPromptTemplate.from_template(
    """Improve the following text based on this feedback:
    
Original text:
{text}

Feedback:
{feedback}

Return JSON with this structure:
{{
  "content": [improved text],
  "change_summary": [what you changed and why]
}}

Return only valid JSON.
"""
)

refine_chain = refine_prompt | llm | StrOutputParser()

print("✓ Feedback and refinement chains created")

In [ ]:
# The Self-Refine Loop
def self_refine_loop(initial_text: str, topic: str, max_iterations: int = 3):
    """
    Iteratively refine text until score >= 8 or max iterations reached.
    """
    current_text = initial_text
    iteration = 0
    
    print(f"\nStarting Self-Refine Loop: {topic}")
    print("=" * 70)
    
    while iteration < max_iterations:
        iteration += 1
        print(f"\n[Iteration {iteration}]")
        
        # Get feedback
        feedback_response = feedback_chain.invoke({"text": current_text})
        feedback_json = json.loads(feedback_response)
        feedback = Feedback(**feedback_json)
        
        print(f"Score: {feedback.score}/10")
        print(f"Issues: {', '.join(feedback.issues) if feedback.issues else 'None'}")
        
        # Check stopping criteria
        if feedback.score >= 8:
            print(f"✓ Score >= 8. Stopping.")
            return current_text, feedback, iteration
        
        if iteration >= max_iterations:
            print(f"✓ Max iterations ({max_iterations}) reached. Stopping.")
            return current_text, feedback, iteration
        
        # Refine if we haven't reached goals
        print(f"Refining...")
        refine_response = refine_chain.invoke({
            "text": current_text,
            "feedback": json.dumps(feedback_json)
        })
        refined_json = json.loads(refine_response)
        refined = RefinedOutput(**refined_json)
        
        print(f"Changes: {refined.change_summary}")
        current_text = refined.content
    
    return current_text, feedback, iteration

# Test the loop
initial_explanation = "An API is a tool that lets software talk. You use it to get data."
final_text, final_feedback, iterations = self_refine_loop(
    initial_text=initial_explanation,
    topic="Explain what an API is",
    max_iterations=3
)

print("\n" + "=" * 70)
print("\nFinal Result:")
print(f"Iterations: {iterations}")
print(f"Final Score: {final_feedback.score}/10")
print(f"\nFinal Text:")
print(final_text)

---
# Section 3: Reflexion Algorithm

A more sophisticated pattern with **memory** and **strategic thinking**:

```
Actor (generates solution)
  ↓
Evaluator (tests solution, provides results)
  ↓
Self-Reflection (analyzes failures, generates insights)
  ↓
Memory (stores reflections)
  ↓
Actor (tries again with lessons learned)
```

The key: the agent **learns from failures** and **remembers lessons**.

In [ ]:
# Pydantic schemas for Reflexion
class AttemptResult(BaseModel):
    """Result from the actor's attempt."""
    code: str = Field(..., description="The generated code")
    explanation: str = Field(..., description="Brief explanation of approach")

class EvaluationResult(BaseModel):
    """Evaluation of whether the code works."""
    passed: bool = Field(..., description="Does the code work correctly?")
    test_results: str = Field(..., description="What tests pass/fail")
    error_message: Optional[str] = Field(
        default=None,
        description="Error description if failed"
    )

class ReflectionInsight(BaseModel):
    """Insights from reflecting on a failure."""
    root_cause: str = Field(..., description="Why did it fail?")
    lesson_learned: str = Field(..., description="What to do differently next time")

print("✓ Reflexion schemas defined")

In [ ]:
# Chain 1: Actor (generates code)
actor_prompt = ChatPromptTemplate.from_template(
    """You are an expert Python programmer. Generate a Python function.

Task: {task}

Previous attempts and what went wrong:
{memory}

Return JSON with this structure:
{{
  "code": [the function code],
  "explanation": [brief explanation of your approach]
}}

Return only valid JSON.
"""
)

actor_chain = actor_prompt | llm | StrOutputParser()

# Chain 2: Evaluator (tests the code)
def evaluate_code(code: str, task: str) -> EvaluationResult:
    """Simple evaluation: try to execute and check for specific test cases."""
    evaluator_prompt = ChatPromptTemplate.from_template(
        """Test this Python function:

Function:
{code}

Task requirements: {task}

Check:
1. Does it have syntax errors?
2. Does it handle basic test cases correctly?
3. Test with a few examples.

Return JSON:
{{
  "passed": [true/false],
  "test_results": [description of what works/fails],
  "error_message": [null or description of error]
}}

Return only valid JSON.
"""
    )
    
    evaluator_chain = evaluator_prompt | llm | StrOutputParser()
    response = evaluator_chain.invoke({"code": code, "task": task})
    
    eval_json = json.loads(response)
    return EvaluationResult(**eval_json)

# Chain 3: Self-Reflection (learns from failures)
reflection_prompt = ChatPromptTemplate.from_template(
    """Analyze why this code failed:

Task: {task}
Code attempted:
{code}

Evaluation:
{evaluation}

Return JSON:
{{
  "root_cause": [why it failed],
  "lesson_learned": [what to do differently in the next attempt]
}}

Return only valid JSON.
"""
)

reflection_chain = reflection_prompt | llm | StrOutputParser()

print("✓ Reflexion chains created")

In [ ]:
# The Reflexion Loop
def reflexion_loop(task: str, max_trials: int = 3):
    """
    Reflexion: Actor → Evaluator → Self-Reflection → Memory → Repeat
    """
    memory = []  # Store reflections
    trial = 0
    
    print(f"\nReflexion Loop: {task}")
    print("=" * 70)
    
    while trial < max_trials:
        trial += 1
        print(f"\n[Trial {trial}]")
        
        # Step 1: Actor generates code
        memory_text = "\n".join(memory) if memory else "(No previous attempts)"
        actor_response = actor_chain.invoke({
            "task": task,
            "memory": memory_text
        })
        actor_json = json.loads(actor_response)
        attempt = AttemptResult(**actor_json)
        
        print(f"Generated code:\n{attempt.code[:150]}...")
        
        # Step 2: Evaluator tests
        print(f"Evaluating...")
        evaluation = evaluate_code(attempt.code, task)
        print(f"Result: {'PASSED ✓' if evaluation.passed else 'FAILED ✗'}")
        
        if evaluation.passed:
            print(f"\n✓ Success! Code works correctly.")
            return attempt.code, trial, memory
        
        # Step 3: Self-Reflection
        print(f"Reflecting on failure...")
        reflection_response = reflection_chain.invoke({
            "task": task,
            "code": attempt.code,
            "evaluation": json.dumps({
                "passed": evaluation.passed,
                "test_results": evaluation.test_results,
                "error_message": evaluation.error_message
            })
        })
        reflection_json = json.loads(reflection_response)
        reflection = ReflectionInsight(**reflection_json)
        
        print(f"Root cause: {reflection.root_cause}")
        print(f"Lesson: {reflection.lesson_learned}")
        
        # Step 4: Update memory
        memory_entry = f"Trial {trial}: {reflection.root_cause}. Lesson: {reflection.lesson_learned}"
        memory.append(memory_entry)
    
    print(f"\n✗ Failed after {max_trials} trials.")
    return None, trial, memory

# Test the Reflexion loop
final_code, num_trials, memory_log = reflexion_loop(
    task="Write a function that returns True if number n is divisible by both 3 and 5, False otherwise. Handle edge cases.",
    max_trials=3
)

print("\n" + "=" * 70)
print("\nMemory Log (what the agent learned):")
for entry in memory_log:
    print(f"  - {entry}")

if final_code:
    print(f"\nFinal working code:")
    print(final_code)

---
# Section 4: LLM-as-Judge Evaluator

Using an LLM to **judge and compare** outputs:
- **Pattern 1**: Single-point scoring with rubric
- **Pattern 2**: Pairwise comparison (which is better?)

In [ ]:
# Pydantic schemas for judging
class JudgeScore(BaseModel):
    """Single-point evaluation with rubric."""
    helpfulness: int = Field(
        ...,
        ge=1,
        le=5,
        description="1-5: How helpful is this response?"
    )
    accuracy: int = Field(
        ...,
        ge=1,
        le=5,
        description="1-5: How accurate/factually correct?"
    )
    clarity: int = Field(
        ...,
        ge=1,
        le=5,
        description="1-5: How clear and well-explained?"
    )
    overall_score: float = Field(
        ...,
        description="Average of the three scores"
    )
    comments: str = Field(
        ...,
        description="Brief evaluation comments"
    )

class PairwiseVerdict(BaseModel):
    """Comparison of two responses."""
    reasoning: str = Field(
        ...,
        description="Detailed analysis before deciding"
    )
    winner: str = Field(
        ...,
        description="'A', 'B', or 'TIE'"
    )
    explanation: str = Field(
        ...,
        description="Why one is better"
    )

print("✓ Judge schemas defined")

In [ ]:
# Pattern 1: Single-point scoring
judge_prompt = ChatPromptTemplate.from_template(
    """Evaluate this response on three dimensions:

Question: {question}

Response:
{response}

Rubric:
- Helpfulness (1-5): Does it directly answer the question? Is it useful?
- Accuracy (1-5): Is the information correct? Any false claims?
- Clarity (1-5): Is it well-written? Easy to understand?

Return JSON:
{{
  "helpfulness": [1-5],
  "accuracy": [1-5],
  "clarity": [1-5],
  "overall_score": [average of three],
  "comments": [your brief evaluation]
}}

Return only valid JSON.
"""
)

judge_chain = judge_prompt | llm | StrOutputParser()

# Test: evaluate two explanations of "machine learning"
ml_response_1 = """Machine learning is when computers learn from data without being explicitly programmed. 
The computer finds patterns in the data and uses them to make predictions on new data."""

ml_response_2 = """Machine learning (ML) is a subset of artificial intelligence that enables systems to learn 
and improve from experience without being explicitly programmed. It uses algorithms to parse data, learn from it, 
and make predictions or decisions based on what it has learned. Common applications include recommendation systems, 
image recognition, and natural language processing."""

print("Evaluating Response 1:")
print("=" * 70)
print(ml_response_1)
print()

score1_response = judge_chain.invoke({
    "question": "What is machine learning?",
    "response": ml_response_1
})
score1_json = json.loads(score1_response)
score1 = JudgeScore(**score1_json)

print(f"Helpfulness: {score1.helpfulness}/5")
print(f"Accuracy: {score1.accuracy}/5")
print(f"Clarity: {score1.clarity}/5")
print(f"Overall: {score1.overall_score:.1f}/5")
print(f"Comments: {score1.comments}")

print("\nEvaluating Response 2:")
print("=" * 70)
print(ml_response_2)
print()

score2_response = judge_chain.invoke({
    "question": "What is machine learning?",
    "response": ml_response_2
})
score2_json = json.loads(score2_response)
score2 = JudgeScore(**score2_json)

print(f"Helpfulness: {score2.helpfulness}/5")
print(f"Accuracy: {score2.accuracy}/5")
print(f"Clarity: {score2.clarity}/5")
print(f"Overall: {score2.overall_score:.1f}/5")
print(f"Comments: {score2.comments}")

In [ ]:
# Pattern 2: Pairwise comparison
pairwise_prompt = ChatPromptTemplate.from_template(
    """Compare these two responses to the same question.

Question: {question}

Response A:
{response_a}

Response B:
{response_b}

First, provide detailed reasoning comparing the two responses.
Then decide which is better (A, B, or TIE) based on:
- Accuracy: Is the information correct?
- Clarity: Which explains better?
- Completeness: Which covers more relevant points?

Return JSON:
{{
  "reasoning": [detailed comparison],
  "winner": ["A", "B", or "TIE"],
  "explanation": [why one is better]
}}

Return only valid JSON.
"""
)

pairwise_chain = pairwise_prompt | llm | StrOutputParser()

# Test with our two responses
print("\nPairwise Comparison:")
print("=" * 70)

pairwise_response = pairwise_chain.invoke({
    "question": "What is machine learning?",
    "response_a": ml_response_1,
    "response_b": ml_response_2
})
pairwise_json = json.loads(pairwise_response)
verdict = PairwiseVerdict(**pairwise_json)

print(f"Winner: Response {verdict.winner}")
print(f"\nReasoning:\n{verdict.reasoning}")
print(f"\nExplanation: {verdict.explanation}")

In [ ]:
# Bias mitigation: Run pairwise comparison twice with swapped order
print("\nBias Mitigation - Running comparison again with swapped order:")
print("=" * 70)

pairwise_response_swapped = pairwise_chain.invoke({
    "question": "What is machine learning?",
    "response_a": ml_response_2,  # Swapped
    "response_b": ml_response_1   # Swapped
})
pairwise_json_swapped = json.loads(pairwise_response_swapped)
verdict_swapped = PairwiseVerdict(**pairwise_json_swapped)

# Convert back to original labeling
winner_original = 'B' if verdict_swapped.winner == 'A' else 'A' if verdict_swapped.winner == 'B' else 'TIE'

print(f"Winner (converted back): Response {winner_original}")
print(f"\nReasoning:\n{verdict_swapped.reasoning}")

# Compare consistency
consistent = (verdict.winner == winner_original) or (verdict.winner == 'TIE' and winner_original == 'TIE')
print(f"\n{'✓' if consistent else '⚠'} Consistency check: {verdict.winner} (run 1) vs {winner_original} (run 2)")

---
# Section 5: Choosing the Right Pattern

## Decision Guide

| Pattern | API Calls | Best For | Cost | Complexity |
|---------|-----------|----------|------|------------|
| **Basic Self-Critique** | 2 (generate + critique) | Simple quality checks | Low | Low |
| **Self-Refine Loop** | 2-6 (multiple rounds) | Iterative improvement, content writing | Medium | Medium |
| **Reflexion** | 3+ per trial | Complex tasks, code generation | High | High |
| **LLM-as-Judge** | 1 (single) or 2+ (pairwise) | Comparing options, ranking | Medium | Low |

## Decision Tree

**Q: Do you need to improve the output?**
- No → Use **LLM-as-Judge** (fast comparison)
- Yes → Continue

**Q: Is iteration enough, or do you need memory of past failures?**
- Just iteration → Use **Self-Refine Loop**
- Need memory/strategic retry → Use **Reflexion**

**Q: Is this a one-shot evaluation?**
- Yes → Use **Basic Self-Critique**
- Multiple rounds needed → Use **Self-Refine** or **Reflexion**

---
# Section 6: Hands-On Exercises

## Exercise 1: Build a Self-Improving Essay Writer

**Goal**: Write a system that takes a topic and automatically improves an essay through multiple iterations.

**Given**:
- A topic string
- A simple initial essay (you'll generate it)

**TODO 1**: Define Pydantic schemas
- `EssayDraft`: contains essay text and word count
- `EssayFeedback`: contains score, weaknesses (list), and improvements (list)

**TODO 2**: Create the essay generation chain
- Prompt: "Write a brief essay on {topic}. Make it 100-150 words."

**TODO 3**: Create the essay feedback chain
- Evaluate on: clarity, argument strength, examples, grammar
- Return structured feedback

**TODO 4**: Implement the refinement loop
- Max 3 iterations
- Stop when score >= 8
- Print each iteration's score and improvements

**Test cases**:
- "The importance of continuous learning"
- "How AI will change the world"

In [ ]:
# Exercise 1: Self-Improving Essay Writer

# TODO 1: Define Pydantic schemas for EssayDraft and EssayFeedback
# Hint: EssayDraft should have fields for essay content and word count
# Hint: EssayFeedback should have score (1-10), weaknesses list, improvements list

class EssayDraft(BaseModel):
    """Your code here"""
    pass

class EssayFeedback(BaseModel):
    """Your code here"""
    pass


# TODO 2: Create the essay generation chain
# Hint: Use ChatPromptTemplate with a template
# Hint: Return the essay as a plain string using StrOutputParser

essay_generation_chain = None  # TODO: Implement


# TODO 3: Create the essay feedback chain
# Hint: Take the essay as input
# Hint: Return JSON with score, weaknesses, improvements
# Hint: Use llm.with_structured_output() or parse JSON manually

essay_feedback_chain = None  # TODO: Implement


# TODO 4: Implement the refinement loop
# Hint: Similar to self_refine_loop() in Section 2
# Hint: Generate → Feedback → Refine → Repeat (max 3 iterations or score >= 8)

def improve_essay(topic: str) -> tuple:
    """
    Generate and iteratively improve an essay.
    
    Args:
        topic: Essay topic
        
    Returns:
        (final_essay, final_feedback, num_iterations)
    """
    # TODO: Implement
    pass


# Test your implementation
print("Testing Exercise 1...")
# TODO: Uncomment and test when ready
# final_essay, feedback, iterations = improve_essay("The importance of continuous learning")
# print(f"Completed in {iterations} iterations with score {feedback.score}")

## Exercise 2: Build an LLM-as-Judge for Code Quality

**Goal**: Compare two code solutions and determine which is better.

**Given**:
- Two code solutions to the same problem
- A problem description

**TODO 1**: Define the `PairwiseCodeJudgment` Pydantic schema
- `reasoning`: detailed comparison
- `winner`: 'A', 'B', or 'TIE'
- `explanation`: why one is better

**TODO 2**: Create the judge chain
- Evaluate on: correctness, readability, efficiency, best practices
- Return structured judgment

**TODO 3**: Implement position-bias mitigation
- Run the judge twice (normal order, then swapped)
- Compare results for consistency
- Report if there's a position bias (one order favors A, other favors B)

**Test cases** (implement simple solutions):
- Problem: "Check if a string is a palindrome"
- Solution A: Naive approach (slice and compare)
- Solution B: Two-pointer approach (optimized)

In [ ]:
# Exercise 2: LLM-as-Judge for Code Quality

# TODO 1: Define the PairwiseCodeJudgment Pydantic schema
# Hint: Should include reasoning, winner ('A'/'B'/'TIE'), and explanation fields

class PairwiseCodeJudgment(BaseModel):
    """Your code here"""
    pass


# TODO 2: Create the judge chain
# Hint: Use ChatPromptTemplate
# Hint: Include explicit rubric in the prompt
# Hint: Evaluate on correctness, readability, efficiency, best practices

code_judge_chain = None  # TODO: Implement


# TODO 3: Implement position-bias mitigation
def judge_code_with_bias_check(problem: str, solution_a: str, solution_b: str) -> dict:
    """
    Judge which code solution is better, checking for position bias.
    
    Args:
        problem: Problem description
        solution_a: First solution
        solution_b: Second solution
        
    Returns:
        dict with 'verdict', 'run1', 'run2', 'consistent' keys
    """
    # TODO: Implement
    # Hint: Run the judge twice (normal and swapped order)
    # Hint: Compare results and detect position bias
    pass


# Test your implementation
print("Testing Exercise 2...")

problem = "Write a function to check if a string is a palindrome."

solution_a = """def is_palindrome(s):
    return s == s[::-1]
"""

solution_b = """def is_palindrome(s):
    left, right = 0, len(s) - 1
    while left < right:
        if s[left] != s[right]:
            return False
        left += 1
        right -= 1
    return True
"""

# TODO: Uncomment and test when ready
# result = judge_code_with_bias_check(problem, solution_a, solution_b)
# print(f"Verdict: {result['verdict']}")
# print(f"Consistent: {result['consistent']}")

---
# Recap

## Patterns Learned

1. **Basic Self-Critique**: Single generate-then-evaluate cycle. Fast, good for sanity checks.

2. **Iterative Self-Refine**: Feedback loop that improves output over multiple rounds. Best for content generation, writing, explanations.

3. **Reflexion Algorithm**: Advanced pattern with memory and strategic reasoning. Agent learns from failures and retries with new insights. Best for complex tasks like code generation.

4. **LLM-as-Judge**: Use LLM to evaluate and compare outputs. Can detect position bias with careful pairwise evaluation.

## Key LangChain Features Used

- **LCEL chains**: `prompt | llm | parser` for composable AI workflows
- **Pydantic BaseModel**: Structured outputs for reliable data extraction
- **ChatPromptTemplate**: Dynamic prompt building with variables
- **StrOutputParser**: Simple string extraction from LLM responses
- **ChatGoogleGenerativeAI**: Google Gemini API integration

## Best Practices

- Always use Pydantic schemas for structured outputs
- Design clear stopping criteria (score thresholds, iteration limits)
- Store memory/reflection for multi-trial loops
- Test for bias in pairwise evaluations
- Use explicit rubrics in evaluation prompts

## Next Session

We'll combine these patterns with **planning and reasoning** to build **fully agentic systems** that can tackle multi-step problems autonomously!

---

© BIA® School of Technology & AI